In [ ]:
# --- Clean + install a stable Sentence-BERT stack for Colab (Py3.12) ---
!pip -q uninstall -y transformers sentence-transformers accelerate datasets peft trl

# Transformers + Accelerate kompatibilis páros
!pip -q install transformers==4.44.2 accelerate==0.33.0

# Sentence-Transformers ehhez passzoló verzió
!pip -q install sentence-transformers==3.0.1

# (opcionális, de hasznos) a gyorsabb letöltés/caching miatt
!pip -q install huggingface-hub==0.24.6

import transformers, accelerate
import sentence_transformers

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("SBERT loaded OK")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

DATA_PATH = "/content/merged_event_logs_dialogue_only.csv"
assert os.path.exists(DATA_PATH), f"File not found at {DATA_PATH}. Upload it to Colab Files."

df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)
print("Columns:", list(df.columns))

def pick_col(candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_text   = pick_col(["content", "dialogue", "text", "utterance"])
col_state  = pick_col(["mental_state", "mentalState", "state"])
col_pers   = pick_col(["personality", "trait"])
col_dpy    = pick_col(["days_per_year", "DAYS_PER_YEAR", "daysPerYear"])
col_deaths = pick_col(["deaths", "DEATHS", "prior_deaths", "trauma_level"])
col_cond   = pick_col(["condition", "Condition", "trauma_condition"])
col_run    = pick_col(["run_id", "seed", "replicate", "run", "simulation_id", "file"])

if col_text is None:
    raise ValueError("Could not find a dialogue text column (expected 'content' or 'dialogue').")

def infer_condition(row):
    if col_cond is not None and pd.notna(row[col_cond]):
        v = str(row[col_cond]).strip().lower()
        if "base" in v or v == "0":
            return "baseline"
        if "trauma" in v or "10000" in v or "10,000" in v:
            return "trauma"
        return v
    if col_deaths is not None and pd.notna(row[col_deaths]):
        try:
            d = int(str(row[col_deaths]).replace(",", "").strip())
            return "baseline" if d == 0 else ("trauma" if d >= 10000 else f"deaths_{d}")
        except:
            return "unknown"
    return "unknown"

df["condition_norm"] = df.apply(infer_condition, axis=1)
df = df[df["condition_norm"].isin(["baseline", "trauma"])].copy()
print("Condition counts:\n", df["condition_norm"].value_counts())

quote_re = re.compile(r'\"(.*)\"')

def extract_dialogue(s: str) -> str:
    if not isinstance(s, str):
        return ""
    m = quote_re.search(s)
    if m:
        return m.group(1).strip()
    s = re.sub(r'^[^:]{0,120}:\s*', '', s).strip()
    return s

def basic_clean(s: str) -> str:
    s = s.replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s

df["dialogue_text"] = df[col_text].astype(str).map(extract_dialogue).map(basic_clean)
df = df[df["dialogue_text"].str.len() > 0].copy()

if col_dpy is None:
    if col_run is not None:
        def infer_dpy_from_run(x):
            s = str(x)
            for k in [1, 12, 52, 365, 356]:
                if re.search(rf'(^|[^0-9]){k}([^0-9]|$)', s):
                    return k
            return np.nan
        df["days_per_year_norm"] = df[col_run].map(infer_dpy_from_run)
    else:
        raise ValueError("No days_per_year column, and no run/file column to infer it from.")
else:
    df["days_per_year_norm"] = pd.to_numeric(df[col_dpy], errors="coerce")

df["days_per_year_norm"] = df["days_per_year_norm"].replace({356: 365})
print("Days-per-year counts:\n", df["days_per_year_norm"].value_counts().sort_index())

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

def embed_texts(texts, batch_size=128):
    return model.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

def centroid_cosine(emb_a, emb_b):
    if len(emb_a) == 0 or len(emb_b) == 0:
        return np.nan
    ca = emb_a.mean(axis=0, keepdims=True)
    cb = emb_b.mean(axis=0, keepdims=True)
    return float(cosine_similarity(ca, cb)[0, 0])

def mean_pairwise_cosine_sampled(emb_a, emb_b, max_pairs=20000, seed=42):
    if len(emb_a) == 0 or len(emb_b) == 0:
        return np.nan
    rng = np.random.default_rng(seed)
    na, nb = len(emb_a), len(emb_b)
    total_pairs = na * nb
    if total_pairs <= max_pairs:
        sims = cosine_similarity(emb_a, emb_b)
        return float(sims.mean())
    ia = rng.integers(0, na, size=max_pairs)
    ib = rng.integers(0, nb, size=max_pairs)
    return float(np.mean(np.sum(emb_a[ia] * emb_b[ib], axis=1)))

overall_rows = []
for dpy, sub in df.groupby("days_per_year_norm"):
    t_base = sub.loc[sub["condition_norm"]=="baseline", "dialogue_text"].tolist()
    t_trau = sub.loc[sub["condition_norm"]=="trauma", "dialogue_text"].tolist()

    e_base = embed_texts(t_base)
    e_trau = embed_texts(t_trau)

    overall_rows.append({
        "days_per_year": int(dpy) if pd.notna(dpy) else dpy,
        "n_baseline": len(t_base),
        "n_trauma": len(t_trau),
        "centroid_cosine": centroid_cosine(e_base, e_trau),
        "mean_pairwise_cosine_sampled": mean_pairwise_cosine_sampled(e_base, e_trau, max_pairs=40000),
    })

overall = pd.DataFrame(overall_rows).sort_values("days_per_year").reset_index(drop=True)
display(overall)

overall_out = "/content/sbert_similarity_overall_by_daysperyear.csv"
overall.to_csv(overall_out, index=False)
print("Saved:", overall_out)

# By mental state (recommended)
if col_state is not None:
    group_cols = ["days_per_year_norm", col_state]
    df["group_key"] = df[group_cols].astype(str).agg("|".join, axis=1)

    bucket = {}
    for (gk, cond), sub in df.groupby(["group_key", "condition_norm"]):
        bucket[(gk, cond)] = sub["dialogue_text"].tolist()

    embeddings = {}
    for (gk, cond), texts in tqdm(bucket.items(), desc="Embedding group buckets"):
        embeddings[(gk, cond)] = embed_texts(texts, batch_size=128)

    rows = []
    for gk in sorted(set(df["group_key"])):
        dpy_s, state_s = gk.split("|", 1)
        emb_base = embeddings.get((gk, "baseline"), np.array([]))
        emb_trau = embeddings.get((gk, "trauma"), np.array([]))

        rows.append({
            "days_per_year": int(float(dpy_s)) if dpy_s.replace('.','',1).isdigit() else dpy_s,
            "mental_state": state_s,
            "n_baseline": len(bucket.get((gk, "baseline"), [])),
            "n_trauma": len(bucket.get((gk, "trauma"), [])),
            "centroid_cosine": centroid_cosine(emb_base, emb_trau),
            "mean_pairwise_cosine_sampled": mean_pairwise_cosine_sampled(emb_base, emb_trau, max_pairs=20000),
        })

    by_state = pd.DataFrame(rows).sort_values(["days_per_year", "mental_state"]).reset_index(drop=True)
    display(by_state.head(50))

    by_state_out = "/content/sbert_similarity_by_state.csv"
    by_state.to_csv(by_state_out, index=False)
    print("Saved:", by_state_out)
else:
    print("No mental_state column detected -> by_state skipped.")